# Holdout A/B: v4.3 vs the senior-dev ports

This notebook runs one or more arms of the holdout A/B on Kaggle's free T4 x2, using the real Gemma 4 31B. The three arms are:

| Arm | Bundle |
| --- | --- |
| `v43` (control) | the best public config (0.10 on the leaderboard) |
| `port` | `bundles/senior-dev-port` |
| `presubmit` | `bundles/senior-dev-presubmit` |

All three share tools, sampling settings (temperature 0.2, thinking off) and budget (4.5 min, 30 calls, 60 turns).

**Before you run:**
1. **Input:** add the competition data with *Add Input → Competitions → Gemma 4 Developer Agent*. Your account must have accepted the rules.
2. **Settings:** set *Accelerator* to **GPU T4 x2**, and turn *Internet* **on**. The notebook downloads llama.cpp, the model and the harness.
3. **Arms:** set `ARMS` in the next cell. One arm over 33 tasks takes about 3.5–4 hours, plus about 40 minutes of setup. Kaggle stops a session at 12 hours, so run **one or two arms per session**. For example, run `["v43", "port"]` in one session and `["presubmit"]` in another.
4. **Run:** use *Save Version → Save & Run All (Commit)*. It runs in the background, and the results land in the version's Output.

**Results:** each arm writes `ab-<arm>-r<repeat>.tgz` to `/kaggle/working`. To get the report, attach the earlier versions' outputs as inputs in a new session and run only the last cell (set `REPORT_ONLY = True`). It finds every `ab-*.tgz` under `/kaggle/input` and `/kaggle/working`.

**How this differs from the scorer:**
- **Serving:** the model runs as the official QAT Q4_0 GGUF on llama.cpp, not W4A16 on vLLM. vLLM can't run W4A16 on T4s. Every arm gets the same model and server, so the comparison is fair. Absolute scores will differ from the leaderboard: v4.3 scored 1/33 here against 0.10 on the leaderboard.
- **Sandboxes:** tasks run in swelite's subprocess sandbox, because there's no Docker on Kaggle.
- **Skills:** stock swelite runs skills differently from the scorer. The notebook patches that with `ab/swelite_official_skills.py`, so the `presubmit` arm calls its skill the way the scorer would.

Built from Gemma-4 commit `b2dfa62`; the harness is pinned to gemma-swe-agent `46b3f66`.

In [ ]:
# ---- settings ----
ARMS = ["port"]          # any of "v43", "port", "presubmit"; one or two per session
REPEAT = 1               # label for this pass; use 2, 3, ... for later passes of the same arm
LIMIT = None             # e.g. 3 for a quick smoke test; None runs all 33 holdout tasks
SPLIT = "holdout"
SESSION_HOURS = 11.5     # stop starting new arms once an arm could not finish inside this
ARM_HOURS_ESTIMATE = 4.2 # per arm on 33 tasks, from earlier T4 x2 runs
REPORT_ONLY = False      # True: skip setup and runs, just build the report from attached results

In [ ]:
import json, os, pathlib, subprocess, sys, tarfile, threading, time, urllib.request
T0 = time.time()
W = pathlib.Path("/kaggle/working")
LOG = open(W / "run.log", "a", buffering=1)
HARNESS_REF = "46b3f66113bba1a6325ea3cc00b9c267e418dc44"
HARNESS = W / "gemma-swe-agent"
BUNDLE_COMMIT = "b2dfa62"

def log(*a):
    s = time.strftime("%H:%M:%S ") + " ".join(str(x) for x in a)
    print(s, flush=True); LOG.write(s + "\n")

def sh(cmd, timeout=None, check=False):
    log("$", cmd[:200])
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if r.stdout.strip(): log(r.stdout[-1500:])
    if r.stderr.strip(): log(r.stderr[-1500:])
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd[:120]}")
    return r

assert set(ARMS) <= {"v43", "port", "presubmit"}, ARMS

In [ ]:
# ---- files embedded from the Gemma-4 repository at commit b2dfa62 ----
EMBEDDED = {
 "bundles/senior-dev-port/README.md": "# senior-dev port\n\nThis is a submission bundle for the Gemma 4 Developer Agent competition. Its prompt is adapted from `/senior-dev`, the bug-fixing subharness in [CodeAF](https://github.com/Agent-Field/CodeAF) (Apache 2.0). The sources are `internal/seniordev/baked/agents/coder.md` and `internal/seniordev/app/solo_prompt.go`.\n\nIt is a draft. It has not been run on any tasks yet.\n\n## Files\n\n| Path | What it holds |\n| --- | --- |\n| `agent.yaml` | One LlmAgent, with the same 7 tools as the v4.3 public config |\n| `prompts/system.md` | The ported prompt |\n| `configs/sampling.yaml` | Temperature 0.2 (senior-dev's setting) with thinking off |\n| `eval_config.yaml` | 4.5 min, 30 tool calls, 60 turns and 180 s: the best-measured public budget |\n\n## What was carried over from senior-dev\n\n| senior-dev | This bundle |\n| --- | --- |\n| One agent in one context, with no planner, reviewer or sub-agents | A single `LlmAgent` with no `sub_agents` or `agent_tool` |\n| Unattended: with no question tool, make a reasonable choice and continue | The same sentence, near the top |\n| The prompt carries mechanics only, never strategy (\"a sentence spent on them is a sentence competing with the repository\") | There is no step-by-step procedure. The sections cover the workspace, the tools, grading, the context window and how the run ends |\n| The request is repeated verbatim at the top | `{problem_description}` comes first, under `# The request` |\n| `.senior-dev/checklist.md` has one `[ ]` line per requirement and `submit` refuses without it | `/tmp/plan.md` has the same checklist and is written in the same call as the reproduction script, so it costs no extra tool call |\n| `.senior-dev/pinned.txt` holds the verify command | The first line of `/tmp/plan.md` is `verify: <command>` |\n| `submit` is the only ending and refuses on an empty tree | `submit_patch` is the only ending. The prompt requires a one-command presubmit check with explicit \"do not submit if\" conditions |\n| After `submit`, senior-dev runs the project's tests itself, so do not edit test or CI configuration | The hidden tests overwrite test files, so pytest.ini, conftest.py and the packaging configs are off limits |\n| An unsubmitted run ships the tree as it stands | True here as well: the fallback diff is graded. The prompt says so |\n| Temperature 0.2 | Temperature 0.2 |\n\n## What was left out, and why\n\n- **Compaction and re-pinning the spec.** senior-dev re-pins `spec.md` after compaction. In the scorer, ADK compaction never fires within a task. It runs only after an invocation completes, and its 32k-token trigger is past vLLM's `max_model_len`. The prompt says nothing is dropped and output must be kept short.\n- **The harness-side refusal.** senior-dev's `submit` tool enforces its checks in code. Here `submit_patch` is fixed, and callbacks come only from a closed registry, so the checks are a self-check the prompt requires. The presubmit variant runs them as one skill script; see below.\n- **The nudges.** senior-dev's continuation messages list facts it observed. The scorer's three nudges are fixed and can't be changed from a bundle.\n- **Web access, grep/glob tools and `apply_patch`.** None exist in the sandbox. Reading uses `grep -n` plus `sed -n` through `run_command`, because the scorer's parser turns `read_file`'s integer arguments into strings.\n\n## How to test it\n\nThe hypothesis under test is that senior-dev's mechanics-only prompt beats the v4.3 procedural prompt on Gemma 4 31B INT4. That is not certain: a 31B model may need the step-by-step procedure that a frontier model does without.\n\n1. Compile the bundle with the local harness (`happyc0der/gemma-swe-agent`, `harness/`):\n   `swelite` or `Compiler(Path(\"bundles/senior-dev-port\"), ...)`.\n2. Run it and the v4.3 control on the same held-out split, with the same seeds. Only the prompt differs, because the tools, sampling and budget match.\n3. Look at the trajectories as well as the score:\n   - Is `/tmp/plan.md` written in the first ~6 calls?\n   - Does the presubmit check run before `submit_patch`?\n   - Did the rate of empty and stray-file diffs go down?\n   - Did the rate of runs ending on the budget or the context window rather than `submit_patch` go down?\n4. If the plan file helps but the free-form exploration doesn't, merge the two: keep v4.3's procedure and add the plan file and presubmit check.\n\n### Presubmit as a skill\n\nBuilt in [`../senior-dev-presubmit/`](../senior-dev-presubmit/). The check lives in `skills/presubmit/SKILL.md` and `scripts/check.py`, declared with `skills: [skills/presubmit]` in `agent.yaml`. The script does the following:\n\n- read `/tmp/plan.md`,\n- run the verify command,\n- print `git status --short` and `git diff --stat`,\n- flag test or config files in the diff and any open `[ ]` lines,\n- end with `READY` or `NOT READY: <reasons>`.\n\nThis is the closest match to senior-dev's refusing `submit`. It still costs one tool call per run.\n",
 "bundles/senior-dev-port/agent.yaml": "# senior-dev port: one agent, one context, no planner, reviewer or sub-agents.\n# Tools match the v4.3 control config exactly, so an A/B against it measures the prompt only.\n# read_file and get_code_subgraph are left out: the scorer's tool-call parser turns their\n# integer/list arguments into strings, so files are read with sed/grep via run_command.\nname: swe_agent\ndescription: Implements one fix in the repository by itself and ends by calling submit_patch.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include prompts/system.md\ngenerate_content_config: !include configs/sampling.yaml\ntools:\n  - run_command\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - search_similar_code\n  - get_code_neighbors\n",
 "bundles/senior-dev-port/configs/sampling.yaml": "# senior-dev's coder agent runs at temperature 0.2. Thinking stays off: it shares the 32k\n# window with tool output, and nothing is compacted within a task.\ntemperature: 0.2\ntop_p: 0.95\ntop_k: 40\nmax_output_tokens: 4096\nthinking_config:\n  include_thoughts: false\n",
 "bundles/senior-dev-port/eval_config.yaml": "# Same budget as the best measured public config (4.5 min scored 0.10; 5.5 min ran past\n# the 12 h cap). timeout_seconds also bounds the hidden-test run, so keep it >= 180.\nevaluation:\n  timeout_seconds: 180\n  max_tool_calls: 30\n  max_time_minutes: 4.5\n  max_turns: 60\n",
 "bundles/senior-dev-port/prompts/system.md": "# The request\n\n{problem_description}\n\n# How this run works\n\nYou are implementing one change in this repository, by yourself, in one context. There is no planner, no reviewer and no one to answer questions: the request above is the whole specification. When something is unclear, make a reasonable choice and continue.\n\nWhat follows is how this run works: the workspace, the tools, what is graded, and what ends the run. How to explore and when to edit are your decisions.\n\n## The workspace\n\n/workspace is a git checkout of the repository at the commit before the fix. Your answer is `git diff HEAD` of /workspace, taken when the run ends. Every file there that differs from HEAD is part of it, including new, untracked and stray files.\n\n/tmp is outside the answer. Keep scratch files there: reproduction scripts, notes, the plan file below. Write them with run_command and a quoted heredoc (`cat > /tmp/x.py <<'EOF'` … `EOF`). write_file only writes inside /workspace, so anything it writes ships.\n\nThe sandbox is offline. pip and network access do not work.\n\n## The tools\n\n- run_command runs bash in /workspace. Each call has a 300 s limit, and stdout and stderr are each cut to 5,000 characters. There is no file-reading tool: read with `grep -n` to find line numbers, then `sed -n 'A,Bp' path` for that region.\n- edit_file takes filepath, old_string and new_string. old_string must match exactly one place. Copy it from fresh `sed -n` output. If it is reported as not found, print the lines again rather than resending the same call.\n- search_similar_code takes one symbol name, such as `HTTPAdapter`, not a sentence. get_code_neighbors lists a symbol's callers, callees and definition.\n- get_status is free. It reports the tool calls, time and turns you have left.\n- Every tool argument is a string. Numeric or boolean options are rejected and waste a call.\n- Python code with quotes or parentheses does not survive `python3 -c` on a command line. Write it to a file in /tmp and run the file.\n\n## The plan file\n\nWrite /tmp/plan.md once, as soon as you know what the fix involves. The same run_command call can also write and run your reproduction script. The file has two parts:\n\n- A first line `verify: <command>` naming the command that shows the fix works. That is usually your reproduction script plus the one or two existing test files that exercise the code you change, for example `python3 /tmp/repro.py && python3 -m pytest tests/test_x.py -x -q 2>&1 | tail -20`.\n- One line per thing the request requires, each starting `[ ] `. Tick one to `[x]` only once the code does it, and change the line with sed.\n\nThe plan file keeps the request's requirements in front of you. Nobody checks it except you, at the end.\n\n## How the answer is graded\n\nA clean copy of the repository gets your diff. Then the original developers' tests for this issue are added and pytest runs. The task counts only if pytest exits 0.\n\nThose tests overwrite any test file with the same path. Editing tests never helps, and new test files can collide with theirs. Change the implementation. Never modify pytest.ini, conftest.py, setup.cfg, pyproject.toml or tox.ini.\n\nExisting tests that passed before your change must still pass after it. A fix that narrows behaviour to the reported case at the cost of existing behaviour is a regression.\n\n## The context window\n\nThe whole run shares one 32k-token window, and nothing in it is summarized or dropped. Every output you request stays until the end, and a run that fills the window stops where it is. Bound every command's output with `| head -30`, `| tail -20`, `grep -m 5`, `-q` and `--tb=short`. Read at most about 80 lines per call, and never run the whole test suite.\n\n## Ending the run\n\nThe run ends when you call submit_patch. Nothing else ends it: no summary, report or status line. The only other ends are the budget running out and three turns in a row without a tool call, and then whatever the tree holds at that moment is graded as it stands.\n\nBefore calling submit_patch, run this check as one command, with your verify command from /tmp/plan.md in place of VERIFY:\n\n`git status --short; git diff --stat; grep -c '^\\[ \\]' /tmp/plan.md; VERIFY`\n\nDo not submit if any of these hold:\n- the diff is empty,\n- `git status` shows a file you did not mean to change or create (restore it with `git checkout -- path` or delete it),\n- the diff touches a test file or one of the config files named above,\n- the verify command fails because of your change,\n- a `[ ]` line is still open (finish it, or delete the line if the request does not in fact require it).\n\nFix what the check shows and run it again. When it is clean, call submit_patch.\n\nAn accepted submit_patch takes the tree as it stands. Anything after it does not count.\n\nBudget is fixed. Check get_status whenever you are unsure how much remains. When fewer than 5 tool calls or about 1 minute remain, stop exploring, make sure a fix is in place, run the check and submit. Submit a non-empty patch, even when you are unsure of it: an empty diff cannot pass.\n",
 "bundles/senior-dev-presubmit/README.md": "# senior-dev port: presubmit variant\n\nThis is the [base port](../senior-dev-port/) plus one skill. The base port asks the agent to run the pre-submit check by hand as a shell one-liner. Here that check is a script, run with one `run_skill_script` call. It is the closest this bundle format gets to senior-dev's `submit` tool, which refuses an unfinished answer in code.\n\nThe only differences from the base port are:\n- the `skills:` entry in `agent.yaml`,\n- `skills/presubmit/`,\n- the \"Ending the run\" section of `prompts/system.md`,\n- two sentences elsewhere in the prompt.\n\nThe tools, sampling settings and budget are unchanged.\n\n## What the check does\n\n`skills/presubmit/scripts/check.py` takes no arguments. Its first line is the verdict, `READY: call submit_patch now` or `NOT READY (n to fix …)`. That keeps the verdict visible even in logs that truncate tool output. A NOT READY output ends with a list of every reason. The reasons it can give:\n\n| Check | Why |\n| --- | --- |\n| The diff is empty | An empty patch cannot pass |\n| Test files (`tests/`, `test_*.py`, `*_test.py`) or `conftest.py`, `pytest.ini`, `setup.cfg`, `pyproject.toml` or `tox.ini` changed | The hidden tests overwrite them, and a changed config can break test collection |\n| Untracked scratch files (`repro*`, `debug*`, `tmp*`, `*.orig`, `*.rej`, `*.log`, …) | Everything in /workspace ships |\n| A changed `.py` file does not compile | This is checked in memory, so no `.pyc` files get written |\n| `/tmp/plan.md` is missing, has no `verify:` line, or has open `[ ]` lines | This is senior-dev's checklist rule |\n| The verify command exits non-zero, runs past 150 s, or creates files in /workspace | The command runs with `pipefail`, so `pytest … \\| tail` still fails when pytest does |\n\nOther new files show up as a note rather than a reason, since a fix can legitimately add a module. Untracked `__pycache__` and `.pytest_cache` are deleted by the check itself. The verify run sets `PYTHONDONTWRITEBYTECODE` and disables pytest's cache.\n\n## Scorer details it relies on\n\nThese come from the official packages (`adk_submission` 0.2.11 and `adk_eval_core`), as vendored in `happyc0der/gemma-swe-agent`:\n\n- **How it's run:** skills run through `AdkSandboxCodeExecutor`. ADK wraps the script and writes the wrapper to `/workspace/.adk_exec_<hex>.py`. It runs the wrapper with `python3` from a temporary directory, then deletes it. The check ignores that file and uses only absolute paths.\n- **Arguments:** the declared parameters are `skill_name`, `file_path`, `args`, `short_options` and `positional_args`. Only the two string ones are used, because the scorer's tool-call parser turns non-string arguments into strings.\n- **Tool-call budget:** each run goes through the budget check, so it costs one tool call, the same as a `run_command`.\n- **Extra prompt:** ADK adds `list_skills`, `load_skill`, `load_skill_resource` and `run_skill_script`, plus about 500 tokens of its own skill instructions. Those instructions tell the model it must call `load_skill` first. The prompt says that's allowed but not needed. `SKILL.md` is kept short, so a `load_skill` call costs a turn and little context.\n\n## What was tested\n\n- The official `compile_submission` and the local `swelite` compiler both accept the bundle.\n- `check.py` was run directly against a scratch git repository in 12 scenarios:\n  - a clean fix,\n  - an empty diff,\n  - a test edit plus a scratch file,\n  - a syntax error,\n  - an open checklist item,\n  - no plan file,\n  - a failing verify behind `| tail`,\n  - a verify command that leaves a file,\n  - only the executor's temporary file present,\n  - a new source file,\n  - stale caches,\n  - a verify timeout.\n\n  Each gave the expected verdict.\n- End to end, the bundle was compiled with the official compiler and `AdkSandboxCodeExecutor` over a stand-in sandbox. `run_skill_script` was then called with the two string arguments. It returned NOT READY before the fix and READY after it, and left no files behind.\n\nNot yet tested: runs with the real model. The question is whether Gemma calls the skill before `submit_patch`, and whether it acts on NOT READY or loops on it.\n\n## What to look at in trajectories\n\n- **Coverage:** the share of runs that call `run_skill_script` before `submit_patch`.\n- **Follow-through:** the share of NOT READY verdicts followed by a fix, and the share followed by an immediate re-run with nothing changed.\n- **Outcomes:** stray-file, test-edit and empty-diff rates, against the base port and v4.3.\n- **Budget:** turns spent on `load_skill` and `list_skills`.\n",
 "bundles/senior-dev-presubmit/agent.yaml": "# senior-dev port, presubmit variant: the base port plus a skill that runs the\n# pre-submit check as one script, standing in for senior-dev's refusing submit tool.\n# Tools, sampling and budget match the base port and the v4.3 control config.\n# read_file and get_code_subgraph are left out: the scorer's tool-call parser turns their\n# integer/list arguments into strings, so files are read with sed/grep via run_command.\nname: swe_agent\ndescription: Implements one fix in the repository by itself and ends by calling submit_patch.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include prompts/system.md\ngenerate_content_config: !include configs/sampling.yaml\ntools:\n  - run_command\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - search_similar_code\n  - get_code_neighbors\nskills:\n  - skills/presubmit\n",
 "bundles/senior-dev-presubmit/configs/sampling.yaml": "# senior-dev's coder agent runs at temperature 0.2. Thinking stays off: it shares the 32k\n# window with tool output, and nothing is compacted within a task.\ntemperature: 0.2\ntop_p: 0.95\ntop_k: 40\nmax_output_tokens: 4096\nthinking_config:\n  include_thoughts: false\n",
 "bundles/senior-dev-presubmit/eval_config.yaml": "# Same budget as the best measured public config (4.5 min scored 0.10; 5.5 min ran past\n# the 12 h cap). timeout_seconds also bounds the hidden-test run, so keep it >= 180.\nevaluation:\n  timeout_seconds: 180\n  max_tool_calls: 30\n  max_time_minutes: 4.5\n  max_turns: 60\n",
 "bundles/senior-dev-presubmit/prompts/system.md": "# The request\n\n{problem_description}\n\n# How this run works\n\nYou are implementing one change in this repository, by yourself, in one context. There is no planner, no reviewer and no one to answer questions: the request above is the whole specification. When something is unclear, make a reasonable choice and continue.\n\nWhat follows is how this run works: the workspace, the tools, what is graded, and what ends the run. How to explore and when to edit are your decisions.\n\n## The workspace\n\n/workspace is a git checkout of the repository at the commit before the fix. Your answer is `git diff HEAD` of /workspace, taken when the run ends. Every file there that differs from HEAD is part of it, including new, untracked and stray files.\n\n/tmp is outside the answer. Keep scratch files there: reproduction scripts, notes, the plan file below. Write them with run_command and a quoted heredoc (`cat > /tmp/x.py <<'EOF'` … `EOF`). write_file only writes inside /workspace, so anything it writes ships.\n\nThe sandbox is offline. pip and network access do not work.\n\n## The tools\n\n- run_command runs bash in /workspace. Each call has a 300 s limit, and stdout and stderr are each cut to 5,000 characters. There is no file-reading tool: read with `grep -n` to find line numbers, then `sed -n 'A,Bp' path` for that region.\n- edit_file takes filepath, old_string and new_string. old_string must match exactly one place. Copy it from fresh `sed -n` output. If it is reported as not found, print the lines again rather than resending the same call.\n- search_similar_code takes one symbol name, such as `HTTPAdapter`, not a sentence. get_code_neighbors lists a symbol's callers, callees and definition.\n- get_status is free. It reports the tool calls, time and turns you have left.\n- Every tool argument is a string. Numeric or boolean options are rejected and waste a call.\n- Python code with quotes or parentheses does not survive `python3 -c` on a command line. Write it to a file in /tmp and run the file.\n\n## The plan file\n\nWrite /tmp/plan.md once, as soon as you know what the fix involves. The same run_command call can also write and run your reproduction script. The file has two parts:\n\n- A first line `verify: <command>` naming the command that shows the fix works. That is usually your reproduction script plus the one or two existing test files that exercise the code you change, for example `python3 /tmp/repro.py && python3 -m pytest tests/test_x.py -x -q 2>&1 | tail -20`.\n- One line per thing the request requires, each starting `[ ] `. Tick one to `[x]` only once the code does it, and change the line with sed.\n\nThe plan file keeps the request's requirements in front of you. The presubmit check reads it at the end.\n\n## How the answer is graded\n\nA clean copy of the repository gets your diff. Then the original developers' tests for this issue are added and pytest runs. The task counts only if pytest exits 0.\n\nThose tests overwrite any test file with the same path. Editing tests never helps, and new test files can collide with theirs. Change the implementation. Never modify pytest.ini, conftest.py, setup.cfg, pyproject.toml or tox.ini.\n\nExisting tests that passed before your change must still pass after it. A fix that narrows behaviour to the reported case at the cost of existing behaviour is a regression.\n\n## The context window\n\nThe whole run shares one 32k-token window, and nothing in it is summarized or dropped. Every output you request stays until the end, and a run that fills the window stops where it is. Bound every command's output with `| head -30`, `| tail -20`, `grep -m 5`, `-q` and `--tb=short`. Read at most about 80 lines per call, and never run the whole test suite.\n\n## Ending the run\n\nThe run ends when you call submit_patch. Nothing else ends it: no summary, report or status line. The only other ends are the budget running out and three turns in a row without a tool call, and then whatever the tree holds at that moment is graded as it stands.\n\nBefore calling submit_patch, run the presubmit check: call run_skill_script with skill_name \"presubmit\" and file_path \"scripts/check.py\", and pass nothing else. Loading the skill first is allowed but not needed. The check reads /tmp/plan.md and runs your verify command itself. It checks that:\n- the diff is not empty,\n- it holds no test files, config files or scratch files,\n- every changed Python file compiles,\n- the verify command passes,\n- no `[ ]` line is still open.\n\nIts first line is READY or NOT READY, and a NOT READY output ends with every reason. On NOT READY, fix what it names and run it again. On READY, call submit_patch.\n\nAn accepted submit_patch takes the tree as it stands. Anything after it does not count.\n\nBudget is fixed. Check get_status whenever you are unsure how much remains. When fewer than 5 tool calls or about 1 minute remain, stop exploring, make sure a fix is in place, run the check once, fix only what is quick, and submit. Submit a non-empty patch, even when you are unsure of it: an empty diff cannot pass.\n",
 "bundles/senior-dev-presubmit/skills/presubmit/SKILL.md": "---\nname: presubmit\ndescription: The check to run right before submit_patch. It reads /tmp/plan.md, checks the diff, and runs the verify command. Its first line is READY or NOT READY, and the reasons come at the end.\n---\nRun it with run_skill_script, passing skill_name \"presubmit\" and file_path \"scripts/check.py\". Pass no other arguments. It costs one tool call.\n\nWhat it checks:\n- /tmp/plan.md exists, has a `verify:` line, and has no open `[ ]` lines.\n- The diff is not empty. It does not touch test files, conftest.py, pytest.ini, setup.cfg, pyproject.toml or tox.ini. It holds no scratch files (repro*, debug*, *.orig, *.rej and similar).\n- Every changed Python file compiles.\n- The verify command exits 0 and leaves no new files in /workspace.\n\nOn READY, call submit_patch. On NOT READY, fix each listed reason and run it again.\n",
 "bundles/senior-dev-presubmit/skills/presubmit/scripts/check.py": "\"\"\"Presubmit check: the bundle's stand-in for senior-dev's refusing submit tool.\n\nReads /tmp/plan.md, inspects `git status` of /workspace, compiles the\nchanged Python files and runs the plan's verify command. The first line is the verdict,\nREADY or NOT READY, and a NOT READY output ends with every reason. It takes no arguments, because the\nscorer's tool-call parser turns non-string arguments into strings.\n\nThe harness runs this file from a temporary copy with a temporary directory as\nthe working directory, so every path here is absolute. It always exits 0: a\nnon-zero exit only adds a traceback to the output.\n\"\"\"\n\nimport os\nimport re\nimport shutil\nimport subprocess\n\nWORKSPACE = os.environ.get(\"PRESUBMIT_WORKSPACE\", \"/workspace\")\nPLAN = os.environ.get(\"PRESUBMIT_PLAN\", \"/tmp/plan.md\")\nVERIFY_TIMEOUT = int(os.environ.get(\"PRESUBMIT_VERIFY_TIMEOUT\", \"150\"))\nTAIL_LINES = 15\n\n# The code executor writes the script it runs into /workspace as .adk_exec_<hex>.py\n# and deletes it afterwards, so it shows up as untracked while this check runs.\nHARNESS_FILE = re.compile(r\"(^|/)\\.adk_exec_[0-9a-f]+\\.py$\")\n# Paths the hidden tests or pytest itself own. The hidden test patch overwrites\n# test files, and a changed config can break collection.\nCONFIG_NAMES = {\"conftest.py\", \"pytest.ini\", \"setup.cfg\", \"pyproject.toml\", \"tox.ini\"}\nTEST_PATH = re.compile(r\"(^|/)tests?/|(^|/)test_[^/]*\\.py$|_test\\.py$\")\n# Caches the check deletes by itself: never tracked, always regenerated.\nCACHE_PATH = re.compile(r\"(^|/)(__pycache__|\\.pytest_cache)/|\\.pyc$\")\n# New files that are almost always scratch work rather than part of a fix.\nSCRATCH_PATH = re.compile(\n    r\"(^|/)(repro|reproduce|debug|scratch|tmp|temp)[^/]*$\"\n    r\"|\\.(orig|rej|bak|swp|log|pyc)$|(^|/)__pycache__/|(^|/)plan\\.md$\"\n)\n\n\ndef git(*args):\n    return subprocess.run(\n        [\"git\", *args], cwd=WORKSPACE, capture_output=True, text=True, errors=\"replace\"\n    )\n\n\ndef changed_paths():\n    \"\"\"Returns (modified, untracked) paths, relative to the workspace.\"\"\"\n    out = git(\"status\", \"--porcelain\", \"--untracked-files=all\").stdout\n    modified, untracked = [], []\n    for line in out.splitlines():\n        if len(line) < 4:\n            continue\n        code, path = line[:2], line[3:]\n        if \" -> \" in path:\n            path = path.split(\" -> \", 1)[1]\n        path = path.strip('\"')\n        if HARNESS_FILE.search(path):\n            continue\n        (untracked if code == \"??\" else modified).append(path)\n    return modified, untracked\n\n\ndef clean_caches():\n    \"\"\"Deletes untracked Python caches, which would otherwise ship. Returns how many.\"\"\"\n    out = git(\"status\", \"--porcelain\", \"--untracked-files=all\").stdout\n    removed = 0\n    for line in out.splitlines():\n        path = line[3:].strip('\"')\n        if not line.startswith(\"??\") or not CACHE_PATH.search(path):\n            continue\n        full = os.path.join(WORKSPACE, path)\n        top = re.match(r\"(.*?(?:__pycache__|\\.pytest_cache))(/|$)\", path)\n        target = os.path.join(WORKSPACE, top.group(1)) if top else full\n        if os.path.isdir(target):\n            shutil.rmtree(target, ignore_errors=True)\n        elif os.path.exists(target):\n            os.remove(target)\n        removed += 1\n    return removed\n\n\ndef read_plan():\n    \"\"\"Returns (verify command or None, open items), or None when the file is missing.\"\"\"\n    try:\n        with open(PLAN, encoding=\"utf-8\", errors=\"replace\") as f:\n            lines = f.read().splitlines()\n    except OSError:\n        return None\n    verify = None\n    open_items = []\n    for line in lines:\n        stripped = line.strip()\n        if verify is None and stripped.lower().startswith(\"verify:\"):\n            verify = stripped[len(\"verify:\"):].strip() or None\n        item = re.sub(r\"^[-*]\\s+\", \"\", stripped)\n        if item.startswith(\"[ ]\"):\n            open_items.append(item[3:].strip())\n    return verify, open_items\n\n\ndef tail(text, n=TAIL_LINES):\n    lines = text.rstrip().splitlines()\n    return \"\\n\".join(lines[-n:])\n\n\nOUT = []\n\n\ndef emit(line=\"\"):\n    OUT.append(line)\n\n\ndef main():\n    reasons = []\n    notes = []\n\n    if not os.path.isdir(os.path.join(WORKSPACE, \".git\")):\n        reasons.append(f\"{WORKSPACE} is not a git checkout\")\n        return reasons, notes\n\n    cleaned = clean_caches()\n    if cleaned:\n        emit(f\"(deleted {cleaned} untracked cache files)\")\n    modified, untracked = changed_paths()\n    changed = modified + untracked\n\n    emit(\"== changes\")\n    if not changed:\n        reasons.append(\"the diff is empty: an empty patch cannot pass\")\n    else:\n        listing = [f\"  M {p}\" for p in modified] + [f\"  + {p}\" for p in untracked]\n        emit(\"\\n\".join(listing[:15]))\n        if len(listing) > 15:\n            emit(f\"  ... and {len(listing) - 15} more\")\n        stat = git(\"diff\", \"--shortstat\", \"HEAD\").stdout.strip()\n        if stat:\n            emit(f\"  ({stat}, not counting new files)\")\n\n    touched_tests = [\n        p for p in changed if TEST_PATH.search(p) or os.path.basename(p) in CONFIG_NAMES\n    ]\n    if touched_tests:\n        reasons.append(\n            \"test or config files changed (hidden tests overwrite them; \"\n            \"restore with git checkout -- PATH, or rm if new): \" + \", \".join(touched_tests)\n        )\n    scratch = [p for p in untracked if SCRATCH_PATH.search(p) and p not in touched_tests]\n    if scratch:\n        reasons.append(\"scratch files inside /workspace ship in the patch; delete them: \" + \", \".join(scratch))\n    other_new = [p for p in untracked if p not in scratch and p not in touched_tests]\n    if other_new:\n        notes.append(\"new files ship in the patch; keep them only if the fix needs them: \" + \", \".join(other_new))\n\n    broken = []\n    for path in changed:\n        full = os.path.join(WORKSPACE, path)\n        if path.endswith(\".py\") and os.path.isfile(full):\n            # compile() in memory: py_compile would write .pyc files into the workspace.\n            try:\n                with open(full, \"rb\") as f:\n                    compile(f.read(), path, \"exec\")\n            except SyntaxError as exc:\n                broken.append(f\"{path}:{exc.lineno}: {exc.msg}\")\n            except ValueError as exc:\n                broken.append(f\"{path}: {exc}\")\n    if broken:\n        reasons.append(\"syntax errors: \" + \"; \".join(broken))\n\n    plan = read_plan()\n    if plan is None:\n        reasons.append(f\"{PLAN} does not exist: write it with a verify: line and one [ ] line per requirement\")\n    else:\n        verify, open_items = plan\n        if open_items:\n            reasons.append(\n                \"open [ ] items in the plan (finish them, or delete a line the request does not require): \"\n                + \"; \".join(open_items)\n            )\n        if verify is None:\n            reasons.append(f\"{PLAN} has no verify: line\")\n        elif not broken:\n            emit(f\"== verify: {verify}\")\n            try:\n                # pipefail: a verify line ending in \"| tail\" must still fail when pytest does.\n                # The env keeps the run from leaving caches in the workspace.\n                env = dict(os.environ, PYTHONDONTWRITEBYTECODE=\"1\")\n                env[\"PYTEST_ADDOPTS\"] = (env.get(\"PYTEST_ADDOPTS\", \"\") + \" -p no:cacheprovider\").strip()\n                res = subprocess.run(\n                    [\"bash\", \"-o\", \"pipefail\", \"-c\", verify], cwd=WORKSPACE, capture_output=True,\n                    text=True, errors=\"replace\", timeout=VERIFY_TIMEOUT, env=env,\n                )\n                output = tail((res.stdout or \"\") + (res.stderr or \"\"))\n                if output:\n                    emit(output)\n                emit(f\"(exit {res.returncode})\")\n                if res.returncode != 0:\n                    reasons.append(\n                        f\"the verify command exits {res.returncode}. If that failure predates \"\n                        \"your change, narrow the verify line to what your change affects\"\n                    )\n            except subprocess.TimeoutExpired:\n                reasons.append(f\"the verify command ran past {VERIFY_TIMEOUT}s; narrow it to specific test files\")\n\n            # The verify command may leave files behind, and they would ship.\n            clean_caches()\n            _, after = changed_paths()\n            leftovers = [p for p in after if p not in untracked]\n            if leftovers:\n                reasons.append(\"the verify command created files in /workspace; delete them: \" + \", \".join(leftovers[:10]))\n\n    return reasons, notes\n\n\ndef render(reasons, notes):\n    \"\"\"Verdict first (log viewers and truncated tool output keep it), details, then reasons.\"\"\"\n    if reasons:\n        lines = [f\"NOT READY ({len(reasons)} to fix, listed at the end)\"]\n    else:\n        lines = [\"READY: call submit_patch now\"]\n    lines += OUT\n    lines += [f\"note: {note}\" for note in notes]\n    if reasons:\n        lines.append(\"NOT READY:\")\n        lines += [f\"  - {reason}\" for reason in reasons]\n    print(\"\\n\".join(lines))\n\n\nif __name__ == \"__main__\":\n    try:\n        render(*main())\n    except Exception as exc:  # the check must never crash the turn\n        print(f\"NOT READY: presubmit check failed to run: {type(exc).__name__}: {exc}\")\n",
 "ab/swelite_official_skills.py": "\"\"\"Make swelite run skill scripts the way the official scorer does.\n\nswelite gives ADK's SkillToolset a sandbox *environment*, so its run_skill_script\ndeclares a required `command` argument. The official scorer (swegemma 0.2.7,\nadk_submission 0.2.11) gives it a *code executor* instead\n(adk_eval_core.sandbox.base.AdkSandboxCodeExecutor). There, run_skill_script takes\nskill_name and file_path, and ADK wraps the script and runs the wrapper as\n`python3 /workspace/.adk_exec_<hex>.py`, charged as one tool call. A bundle whose\nprompt follows the official contract fails every skill call under stock swelite.\nThis module swaps in an executor with the official behaviour.\n\nImport it before swelite's CLI runs:\n    python -c \"import swelite_official_skills; from swelite.cli import app; app()\" eval ...\n\"\"\"\n\nimport json\nimport uuid\nfrom typing import Any\n\nfrom google.adk.code_executors.base_code_executor import BaseCodeExecutor\nfrom google.adk.code_executors.code_execution_utils import CodeExecutionInput, CodeExecutionResult\nfrom google.adk.skills import load_skill_from_dir\nfrom google.adk.tools.skill_toolset import SkillToolset\nfrom pydantic import ConfigDict\n\nimport swelite.skills\n\nclass SwegemmaLikeExecutor(BaseCodeExecutor):\n    \"\"\"Mirrors AdkSandboxCodeExecutor: write the wrapper into /workspace, run it with python3, delete it.\"\"\"\n\n    model_config = ConfigDict(arbitrary_types_allowed=True)\n    ctx: Any = None\n\n    def execute_code(self, invocation_context: Any, code_execution_input: CodeExecutionInput) -> CodeExecutionResult:\n        name = f\".adk_exec_{uuid.uuid4().hex[:8]}.py\"\n        self.ctx.sandbox.write_text(f\"/workspace/{name}\", code_execution_input.code)\n        # The subprocess sandbox maps /workspace and /tmp to private directories by rewriting\n        # command strings, not file contents, so check.py learns the real paths from this\n        # command. Its verify line needs nothing: the agent wrote the plan file through\n        # run_command, so the sandbox already rewrote the paths in it.\n        command = f\"PRESUBMIT_WORKSPACE=/workspace PRESUBMIT_PLAN=/tmp/plan.md python3 /workspace/{name}\"\n        try:\n            # ctx.run_command gates on the budget and charges one tool call, as the scorer does.\n            raw = json.loads(self.ctx.run_command(command))\n        finally:\n            self.ctx.sandbox.exec(f\"rm -f /workspace/{name}\", timeout=30)\n        if raw.get(\"status\") == \"ok\":\n            return CodeExecutionResult(stdout=raw.get(\"stdout\", \"\"), stderr=\"\", output_files=[])\n        details = raw.get(\"details\", {})\n        stderr = details.get(\"stderr\") or raw.get(\"error_message\", \"\")\n        return CodeExecutionResult(stdout=details.get(\"stdout\", raw.get(\"stdout\", \"\")), stderr=stderr, output_files=[])\n\n\ndef build_skill_toolset(skill_dirs, env, script_timeout=300):\n    skills = [load_skill_from_dir(d) for d in skill_dirs]\n    return SkillToolset(skills=skills, code_executor=SwegemmaLikeExecutor(ctx=env.ctx), script_timeout=script_timeout)\n\n\nswelite.skills.build_skill_toolset = build_skill_toolset\n",
 "ab/report.py": "\"\"\"Summarise an A/B run from ab/run_holdout_ab.sh.\n\nUsage: python3 ab/report.py ab/results/<run> [--control v43]\n\nReads each <arm>-r<n>/ directory swelite wrote (task_results.jsonl, patches/, logs/)\nand prints, per arm: resolved rate, a paired comparison against the control, and the\nbehaviours the senior-dev port is meant to change.\n\"\"\"\n\nimport argparse\nimport json\nimport math\nimport re\nimport sys\nfrom collections import defaultdict\nfrom pathlib import Path\n\nCALL = re.compile(r\"^--- CALL (\\w+) (.*)$\")\nRESP = re.compile(r\"^--- RESP (\\w+) (.*)$\")\nTEST_OR_CONFIG = re.compile(\n    r\"^(tests?/|.*/tests?/|.*(^|/)test_[^/]*\\.py$|.*_test\\.py$|.*(^|/)(conftest\\.py|pytest\\.ini|setup\\.cfg|pyproject\\.toml|tox\\.ini)$)\"\n)\nSCRATCH = re.compile(r\"(^|/)(repro|reproduce|debug|scratch|tmp|temp)[^/]*$|\\.(orig|rej|bak|log|pyc)$|(^|/)plan\\.md$\")\n\n\ndef patch_files(text):\n    \"\"\"Returns (paths, new paths) from a git diff.\"\"\"\n    paths, new = [], []\n    current = None\n    for line in text.splitlines():\n        if line.startswith(\"diff --git \"):\n            current = line.split(\" b/\", 1)[-1]\n            paths.append(current)\n        elif line.startswith(\"new file mode\") and current:\n            new.append(current)\n    return paths, new\n\n\ndef log_features(path):\n    \"\"\"Tool-use facts from one swelite log.\"\"\"\n    f = dict(calls=0, plan_call=None, presubmit_calls=0, ready=0, not_ready=0,\n             presubmit_before_submit=False, manual_check_before_submit=False,\n             load_skill=0, repeat_presubmit_unchanged=0)\n    if not path.exists():\n        return f\n    last_action_was_presubmit = False\n    submitted = False\n    for line in path.read_text(errors=\"replace\").splitlines():\n        m = CALL.match(line)\n        if m:\n            name, args = m.groups()\n            if name in (\"load_skill\", \"list_skills\"):\n                f[\"load_skill\"] += 1\n                continue\n            f[\"calls\"] += 1\n            if f[\"plan_call\"] is None and \"/tmp/plan.md\" in args and name == \"run_command\":\n                f[\"plan_call\"] = f[\"calls\"]\n            if name == \"run_skill_script\":\n                f[\"presubmit_calls\"] += 1\n                if last_action_was_presubmit:\n                    f[\"repeat_presubmit_unchanged\"] += 1\n                last_action_was_presubmit = True\n                if not submitted:\n                    f[\"presubmit_before_submit\"] = True\n                continue\n            if name == \"run_command\" and \"git status\" in args and \"git diff\" in args and not submitted:\n                f[\"manual_check_before_submit\"] = True\n            if name == \"submit_patch\":\n                submitted = True\n            last_action_was_presubmit = False\n            continue\n        m = RESP.match(line)\n        if m and m.group(1) == \"run_skill_script\":\n            body = m.group(2)\n            if \"NOT READY\" in body:\n                f[\"not_ready\"] += 1\n            elif \"READY\" in body:\n                f[\"ready\"] += 1\n    return f\n\n\ndef load_arm(dirpath):\n    rows = {}\n    results = dirpath / \"task_results.jsonl\"\n    if not results.exists():\n        return rows\n    for line in results.read_text().splitlines():\n        if not line.strip():\n            continue\n        row = json.loads(line)\n        iid = row[\"instance_id\"]\n        patch = (dirpath / \"patches\" / f\"{iid}.patch\")\n        text = patch.read_text(errors=\"replace\") if patch.exists() else \"\"\n        paths, new = patch_files(text)\n        row[\"touched_tests\"] = any(TEST_OR_CONFIG.match(p) for p in paths)\n        row[\"scratch\"] = any(SCRATCH.search(p) for p in new)\n        row[\"empty\"] = not text.strip()\n        row.update(log_features(dirpath / \"logs\" / f\"{iid}.log\"))\n        rows[iid] = row\n    return rows\n\n\ndef mcnemar_exact(b, c):\n    \"\"\"Two-sided exact McNemar p-value for discordant counts b and c.\"\"\"\n    n = b + c\n    if n == 0:\n        return 1.0\n    k = min(b, c)\n    p = sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n\n    return min(1.0, 2 * p)\n\n\ndef pct(num, den):\n    return f\"{100 * num / den:5.1f}%\" if den else \"   n/a\"\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"run_dir\", type=Path)\n    ap.add_argument(\"--control\", default=\"v43\")\n    a = ap.parse_args()\n\n    arms = defaultdict(dict)  # arm -> repeat -> rows\n    for d in sorted(a.run_dir.iterdir()):\n        m = re.fullmatch(r\"(.+)-r(\\d+)\", d.name)\n        if d.is_dir() and m:\n            rows = load_arm(d)\n            if rows:\n                arms[m.group(1)][int(m.group(2))] = rows\n    if not arms:\n        sys.exit(f\"no results under {a.run_dir}\")\n\n    print(f\"run: {a.run_dir.name}\")\n    header = f\"{'arm':<11}{'resolved':>14}{'submitted':>11}{'empty':>8}{'tests/cfg':>11}{'scratch':>9}{'plan@≤6':>9}{'checked':>9}\"\n    print(header)\n    print(\"-\" * len(header))\n    for arm in sorted(arms, key=lambda x: (x != a.control, x)):\n        rows = [r for rep in arms[arm].values() for r in rep.values()]\n        n = len(rows)\n        solved = sum(bool(r.get(\"resolved\")) for r in rows)\n        checked = sum(r[\"presubmit_before_submit\"] or r[\"manual_check_before_submit\"] for r in rows)\n        print(f\"{arm:<11}{solved:>5}/{n:<3}{pct(solved, n):>6}\"\n              f\"{pct(sum(bool(r.get('patch_submitted')) for r in rows), n):>11}\"\n              f\"{pct(sum(r['empty'] for r in rows), n):>8}\"\n              f\"{pct(sum(r['touched_tests'] for r in rows), n):>11}\"\n              f\"{pct(sum(r['scratch'] for r in rows), n):>9}\"\n              f\"{pct(sum(r['plan_call'] is not None and r['plan_call'] <= 6 for r in rows), n):>9}\"\n              f\"{pct(checked, n):>9}\")\n\n    if \"presubmit\" in arms:\n        rows = [r for rep in arms[\"presubmit\"].values() for r in rep.values()]\n        runs = sum(r[\"presubmit_calls\"] for r in rows)\n        print(f\"\\npresubmit skill: {runs} runs over {len(rows)} tasks; \"\n              f\"READY {sum(r['ready'] for r in rows)}, NOT READY {sum(r['not_ready'] for r in rows)}, \"\n              f\"re-run with no action between {sum(r['repeat_presubmit_unchanged'] for r in rows)}, \"\n              f\"load/list_skills turns {sum(r['load_skill'] for r in rows)}\")\n\n    if a.control in arms:\n        print(f\"\\npaired vs {a.control} (same task, same repeat; exact McNemar):\")\n        for arm in sorted(arms):\n            if arm == a.control:\n                continue\n            b = c = pairs = 0\n            for rep, rows in arms[arm].items():\n                ctrl = arms[a.control].get(rep, {})\n                for iid, r in rows.items():\n                    if iid not in ctrl:\n                        continue\n                    pairs += 1\n                    x, y = bool(r.get(\"resolved\")), bool(ctrl[iid].get(\"resolved\"))\n                    b += x and not y\n                    c += y and not x\n            print(f\"  {arm:<10} pairs={pairs:<4} wins={b:<3} losses={c:<3} net={b - c:+d}  p={mcnemar_exact(b, c):.3f}\")\n        print(\"  (on ~33 tasks a net of 1-2 is within run-to-run noise; look for p < 0.1 across repeats)\")\n\n\nif __name__ == \"__main__\":\n    main()\n"
}
AB = W / "ab_files"
for rel, text in EMBEDDED.items():
    p = AB / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding="utf-8")
log(f"wrote {len(EMBEDDED)} embedded files under {AB}")

In [ ]:
# ---- checks: data, GPU, disk ----
if not REPORT_ONLY:
    found = subprocess.run("find /kaggle/input -maxdepth 4 -name tasks.jsonl 2>/dev/null | head -1",
                           shell=True, capture_output=True, text=True).stdout.strip()
    assert found, "Competition data not found: add it via Add Input -> Competitions -> Gemma 4 Developer Agent"
    DATA = str(pathlib.Path(found).parent); log("DATA =", DATA)
    gpus = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader").stdout
    assert "T4" in gpus or "L4" in gpus, "Set Accelerator to GPU T4 x2 in the notebook settings"
    sh("df -h /tmp /kaggle/working | tail -2; free -g | head -2")

In [ ]:
# ---- harness: gemma-swe-agent's swelite, pinned ----
if not REPORT_ONLY:
    if not HARNESS.exists():
        sh(f"git clone -q https://github.com/happyc0der/gemma-swe-agent.git {HARNESS}", check=True)
    sh(f"cd {HARNESS} && git fetch -q origin {HARNESS_REF} 2>/dev/null; git checkout -q {HARNESS_REF} && git log --oneline -1", check=True)
    sh(f"{sys.executable} -m pip install -q -e {HARNESS}/harness virtualenv 2>&1 | grep -viE 'warning|incompatible' | tail -2")
    sh(f"{sys.executable} -c 'import google.adk, swelite; print(\"adk\", google.adk.__version__)'", check=True)
    # The presubmit arm needs the scorer's skill behaviour; check the patch imports cleanly.
    sh(f"cd {AB}/ab && {sys.executable} -c 'import swelite_official_skills; print(\"skill patch ok\")'", check=True)
    SPLITS = json.load(open(HARNESS / "experiments" / "splits.json"))
    IDS = [i for i in SPLITS[SPLIT] if i not in SPLITS["env_unstable"]][: LIMIT or None]
    log(f"{SPLIT}: {len(IDS)} tasks")
    ARM_DIRS = {"v43": HARNESS / "experiments/variants/v43",
                "port": AB / "bundles/senior-dev-port",
                "presubmit": AB / "bundles/senior-dev-presubmit"}
    for arm in ARMS:
        assert (ARM_DIRS[arm] / "agent.yaml").exists(), ARM_DIRS[arm]

In [ ]:
# ---- llama.cpp server, built for T4 (sm_75) ----
if not REPORT_ONLY:
    sh("mkdir -p /tmp/cudalib && d=$(dirname $(find /usr/lib /usr/local -name 'libcuda.so.1' 2>/dev/null | head -1)) && ln -sf $d/libcuda.so.1 /tmp/cudalib/libcuda.so")
    if not pathlib.Path("/tmp/llama.cpp/build/bin/llama-server").exists():
        sh("cd /tmp && rm -rf llama.cpp && git clone -q --depth 1 https://github.com/ggml-org/llama.cpp", check=True)
        sh("cd /tmp/llama.cpp && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF "
           "-DCMAKE_LIBRARY_PATH='/tmp/cudalib;/usr/local/cuda/lib64/stubs' -DCMAKE_CUDA_COMPILER=/usr/local/cuda/bin/nvcc > /dev/null 2>&1; "
           "cmake --build build --config Release -j 4 --target llama-server 2>&1 | tail -1", timeout=3600)
    LLAMA_REF = sh("git -C /tmp/llama.cpp log --oneline -1").stdout.strip()
    assert pathlib.Path("/tmp/llama.cpp/build/bin/llama-server").exists(), "llama.cpp build failed; see run.log"

In [ ]:
# ---- weights: official QAT Q4_0 GGUF ----
if not REPORT_ONLY:
    os.environ["HF_HOME"] = "/tmp/hf"
    try:  # only needed if the repository is gated for your account
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        log("using HF_TOKEN from Kaggle secrets")
    except Exception:
        pass
    sh(f"{sys.executable} -m pip install -q huggingface_hub 2>&1 | tail -1")
    from huggingface_hub import hf_hub_download
    t = time.time()
    GGUF = hf_hub_download("google/gemma-4-31B-it-qat-q4_0-gguf", "gemma-4-31B_q4_0-it.gguf", local_dir="/tmp/gguf")
    log(f"gguf ready in {time.time() - t:.0f}s: {GGUF}")

In [ ]:
# ---- serve: one 32k slot (two overflow the T4s), no mmap (its page cache counts against
# Kaggle's 30 GB memory limit and got the server killed), restarted by a watchdog if it dies ----
SERVED = "gemma-4-31b-it-qat-w4a16-ct"
if not REPORT_ONLY:
    SERVER_CMD = ["stdbuf", "-oL", "-eL", "/tmp/llama.cpp/build/bin/llama-server", "-m", GGUF, "-ngl", "999", "-sm", "layer",
                  "-c", "32768", "-np", "1", "--jinja", "--host", "127.0.0.1", "--port", "8000", "--alias", SERVED,
                  "-fa", "on", "--reasoning-format", "auto", "--threads-http", "8"]

    def start_server(tag):
        srv = subprocess.Popen(SERVER_CMD, stdout=open(W / f"llama-{tag}.log", "a"), stderr=subprocess.STDOUT,
                               env={**os.environ, "LLAMA_ARG_MMAP": "0", "LLAMA_ARG_NO_MMAP": "1"})
        for i in range(80):
            time.sleep(15)
            try:
                urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=5).read()
                log(f"llama-server up ({tag}) after {(i + 1) * 15}s"); return srv
            except Exception:
                if srv.poll() is not None:
                    log(f"llama-server died at startup ({tag}):", (W / f"llama-{tag}.log").read_text()[-1500:]); return None
        return None

    server = start_server("0")
    assert server is not None, "llama-server did not start; see llama-0.log"
    RESTARTS = 0

    def watchdog():
        global server, RESTARTS
        while True:
            time.sleep(20)
            if server.poll() is not None:
                RESTARTS += 1
                log(f"WATCHDOG: llama-server exited rc={server.returncode}; restart #{RESTARTS}")
                server = start_server(str(RESTARTS))
                if server is None:
                    log("WATCHDOG: restart failed"); return

    threading.Thread(target=watchdog, daemon=True).start()

In [ ]:
# ---- run the arms ----
# swelite runs with the skill patch imported, so run_skill_script behaves as on the scorer.
ENTRY = f"import sys; sys.path.insert(0, {str(AB / 'ab')!r}); import swelite_official_skills; from swelite.cli import app; app()"
if not REPORT_ONLY:
    (W / "results").mkdir(exist_ok=True)
    for arm in ARMS:
        hours_used = (time.time() - T0) / 3600
        need = ARM_HOURS_ESTIMATE * len(IDS) / 33
        if hours_used + need > SESSION_HOURS:
            log(f"SKIP {arm}: {hours_used:.1f} h used + ~{need:.1f} h needed > {SESSION_HOURS} h; run it in another session")
            continue
        name = f"{arm}-r{REPEAT}"
        out = W / "results" / name
        # Budgets come from each bundle's eval_config.yaml; nothing is overridden here.
        cmd = [sys.executable, "-c", ENTRY, "eval", "--data-dir", DATA, "--submission-dir", str(ARM_DIRS[arm]),
               "--results-dir", str(out), "--sandbox", "subprocess", "--concurrency", "1",
               "--api-base", "http://127.0.0.1:8000/v1", "--served-model", SERVED]
        for i in IDS:
            cmd += ["--task-id", i]
        log(f"START {name}: {len(IDS)} tasks")
        t = time.time()
        with open(W / f"{name}.console", "w") as console:
            rc = subprocess.run(cmd, cwd=HARNESS / "harness", stdout=console, stderr=subprocess.STDOUT).returncode
        log(f"END {name}: rc={rc} in {(time.time() - t) / 60:.0f} min; server restarts so far {RESTARTS}")
        (out / "run.json").write_text(json.dumps({"arm": arm, "repeat": REPEAT, "split": SPLIT, "tasks": len(IDS),
            "bundles_commit": BUNDLE_COMMIT, "harness_ref": HARNESS_REF, "llama_cpp": LLAMA_REF, "gguf": GGUF,
            "minutes": round((time.time() - t) / 60, 1), "server_restarts": RESTARTS}, indent=1))
        with tarfile.open(W / f"ab-{name}.tgz", "w:gz") as tar:
            tar.add(out, arcname=name, filter=lambda ti: None if "/test_outputs/" in ti.name else ti)
        rows = [json.loads(l) for l in (out / "task_results.jsonl").read_text().splitlines() if l.strip()] if (out / "task_results.jsonl").exists() else []
        log(f"RESULT {name}: {sum(bool(r.get('resolved')) for r in rows)}/{len(rows)} resolved -> ab-{name}.tgz")
    server.terminate()
    log(f"RUN_DONE after {(time.time() - T0) / 3600:.1f} h")

In [ ]:
# ---- report: every ab-*.tgz attached as input or produced in this session ----
import shutil
REPORT = W / "ab_report"
shutil.rmtree(REPORT, ignore_errors=True)
# One tarball per arm and repeat. Attached inputs come first, so a tarball made in this
# session replaces an attached one with the same name instead of mixing into it.
chosen = {}
for tgz in sorted(pathlib.Path("/kaggle/input").rglob("ab-*.tgz")) + sorted(W.glob("ab-*.tgz")):
    if tgz.name in chosen:
        log(f"WARNING: two copies of {tgz.name}; using {tgz}, ignoring {chosen[tgz.name]}")
    chosen[tgz.name] = tgz
for tgz in chosen.values():
    with tarfile.open(tgz) as tar:
        tar.extractall(REPORT)
arms_found = sorted(p.name for p in REPORT.iterdir() if p.is_dir()) if REPORT.exists() else []
log("results found:", arms_found or "none")
if arms_found:
    r = subprocess.run([sys.executable, str(AB / "ab" / "report.py"), str(REPORT)], capture_output=True, text=True)
    print(r.stdout or r.stderr)
    (W / "ab_report.txt").write_text(r.stdout or r.stderr)